# FinSight-RAG — Phase 2 (Partially Working System)
**A Source-Grounded Question-Answering Assistant for Corporate Financial Filings**  
Lab 9 PSIS · Activity 1: RAG-Based Domain Assistant · Financial Report Analyzer

This notebook reproduces the Phase 2 evidence: load → chunk → embed → Chroma → retrieve → grounded prompt, plus the test suite. Runtime: CPU is sufficient.

## 1. Setup — replace `<your-username>` with the group's GitHub account

In [11]:
!git clone https://github.com/akashgoyalll/finsight-rag.git
%cd finsight-rag
!pip install -q -r requirements.txt
!bash scripts/get_data.sh

fatal: destination path 'finsight-rag' already exists and is not an empty directory.
/content/finsight-rag/finsight-rag
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.0/572.0 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.2/313.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.3 MB/s eta 0:00:00
   ━━━

## 2. Retrieval evaluation over the 6 test questions (semantic embeddings: all-MiniLM-L6-v2)
No LLM is called. Each question has ground truth verified from the filing; HIT means the retrieved chunks contain the evidence needed to answer.

In [12]:
!EMBEDDINGS=hf python scripts/run_retrieval_demo.py

modules.json: 100% 349/349 [00:00<00:00, 1.25MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 467kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 8.80MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 77.5kB/s]
config.json: 100% 612/612 [00:00<00:00, 1.67MB/s]

model.safetensors: downloading bytes:  94% 85.0M/90.9M [00:01<00:00, 97.6MB/s, 7.39MB/s  ]
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:01<00:00, 62.4MB/s, 8.04MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:01<00:00, 66.7MB/s, 8.77MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 7139.65it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.37MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 16.4MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 60.3MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 397kB/s]
config.json: 100% 190/190 [00:00<00:00, 774kB/s]
Embeddings backend : hf
Pages loaded       : 107  (nke-10k-2023.pdf)
Chunks created     : 516  (size=1

## 3. Offline lexical baseline (TF-IDF) for comparison

In [13]:
!EMBEDDINGS=tfidf python scripts/run_retrieval_demo.py

Embeddings backend : tfidf
Pages loaded       : 107  (nke-10k-2023.pdf)
Chunks created     : 516  (size=1000, overlap=200)
Sample chunk meta  : {'source': 'nke-10k-2023.pdf', 'page': 16, 'total_pages': 107, 'start_index': 3705}
Index build time   : 19.0s   | retriever top-k = 4

Q1 [numeric lookup]
   Q: What were NIKE, Inc.'s total revenues in fiscal 2023?
   truth : $51.2 billion ($51,217 million), +10% reported / +16% currency-neutral
   retrieved pages: [42, 43, 41, 40]   -> MISS (evidence NOT found)
Q2 [factual lookup]
   Q: How many employees did NIKE have as of May 31, 2023?
   truth : approximately 83,700 employees worldwide
   retrieved pages: [28, 9, 79, 9]   -> HIT  (evidence found: 83,700)
Q3 [numeric + explanation]
   Q: How did gross margin change in fiscal 2023 compared to fiscal 2022?
   truth : decreased 250 bps to 43.5% from 46.0%
   retrieved pages: [37, 38, 48, 47]   -> HIT  (evidence found: 250 basis points)
Q4 [numeric lookup]
   Q: How much did NIKE Direct revenu

## 4. Test suite — metadata preservation, retriever, parser, source resolution, LCEL + memory wiring

In [14]:
!python -m pytest tests/ -v -W ignore::DeprecationWarning

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/finsight-rag/finsight-rag
plugins: langsmith-0.12.1, anyio-4.14.2, typeguard-4.6.0
collected 6 items                                                              

tests/test_pipeline.py::test_loader_attaches_source_and_page PASSED      [ 16%]
tests/test_pipeline.py::test_metadata_survives_chunking PASSED           [ 33%]
tests/test_pipeline.py::test_retriever_returns_cited_chunks PASSED       [ 50%]
tests/test_pipeline.py::test_parser_rejects_malformed_output PASSED      [ 66%]
tests/test_pipeline.py::test_sources_resolved_from_metadata_not_llm PASSED [ 83%]
tests/test_pipeline.py::test_full_lcel_chain_and_memory_wiring PASSED    [100%]

========================= 6 passed in 75.32s (0:01:15) =========================


## 5. (Phase 3) Full grounded answers — requires an LLM API key
Install your provider's LangChain package and set `LLM_MODEL` in `<provider>:<model>` format.

In [15]:
# !pip install -q langchain-<provider>
# import os
# os.environ['LLM_MODEL'] = '<provider>:<model>'
# os.environ['<PROVIDER>_API_KEY'] = '...'
# !python scripts/ask.py "What were NIKE's total revenues in fiscal 2023?"